In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)


In [3]:
df = pd.read_csv('../datasets/final_dataset_missing_value_imputed_v5.csv').drop(columns=['store room', 'society', 'price_per_sqft', 'balcony', 'property_id', 'study room', 'pooja room', 'others'])
df.head()


,property_type,sector,price,bedRoom,bathroom,floorNum,agePossession,builtup_area,servant room,furnishing_type,facility_score
0,house,sector 25,4.35,5,4,3.0,Old Property,1350.0,1,0,49
1,flat,sector 50,3.10,3,4,4.0,Moderately Old,2450.0,1,2,165
2,flat,sector 63a,1.65,2,2,1.0,Under Construction,978.0,0,0,0
3,flat,sector 95,2.00,3,4,5.0,New Property,1964.0,1,0,49
4,flat,sector 111,3.90,4,5,6.0,Relatively New,2996.0,1,1,160


will perform linear regression to gain the relation knowlede.

In [4]:
# numerical - bedRoom, bathroom, builtup_area, servant_room
# ordinal - property_type, furnishing_type, luxury_category
# ohe - sector, agePossession


In [5]:
df['agePossession'].value_counts()


agePossession
Relatively New        1723
Moderately Old         618
New Property           595
Old Property           330
Under Construction     276
Name: count, dtype: int64

value counts has 5 types reducing them to 3: old, new, under construction

In [6]:
df['agePossession'].replace({
    'Relatively New': 'new',
    'Moderately Old': 'old',
    'New Property': 'new',
    'Old Property': 'old',
    'Under Construction': 'under_construction',
}, inplace=True)


C:\Users\hp\AppData\Local\Temp\ipykernel_18824\80852311.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['agePossession'].replace({


In [7]:
df['agePossession'].value_counts()


agePossession
new                   2318
old                    948
under_construction     276
Name: count, dtype: int64

In [8]:
df.head()


,property_type,sector,price,bedRoom,bathroom,floorNum,agePossession,builtup_area,servant room,furnishing_type,facility_score
0,house,sector 25,4.35,5,4,3.0,old,1350.0,1,0,49
1,flat,sector 50,3.10,3,4,4.0,old,2450.0,1,2,165
2,flat,sector 63a,1.65,2,2,1.0,under_construction,978.0,0,0,0
3,flat,sector 95,2.00,3,4,5.0,new,1964.0,1,0,49
4,flat,sector 111,3.90,4,5,6.0,new,2996.0,1,1,160


In [9]:
df['property_type'].replace({
    'flat': 0,
    'house': 1
}, inplace=True)


C:\Users\hp\AppData\Local\Temp\ipykernel_18824\1548980210.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['property_type'].replace({
C:\Users\hp\AppData\Local\Temp\ipykernel_18824\1548980210.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type'].replace({


In [10]:
def categorize_facility_score(score):
    if(0 <= score < 50):
        return "Low"
    elif(50 <= score < 150):
        return "Medium"
    elif 150 <= score < 175:
        return "High"
    else:
        return None

# converting facility_score to categorical values
df['facility_category'] = df['facility_score'].apply(categorize_facility_score)
df.drop(columns=['facility_score'], inplace=True)

# converting facility_category to numerical values
df['facility_category'].replace({
    'Low': 0,
    'Medium': 1,
    'High': 2
}, inplace=True)

df.head()


C:\Users\hp\AppData\Local\Temp\ipykernel_18824\3250177137.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['facility_category'].replace({
C:\Users\hp\AppData\Local\Temp\ipykernel_18824\3250177137.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['facility_category'].replace({


,property_type,sector,price,bedRoom,bathroom,floorNum,agePossession,builtup_area,servant room,furnishing_type,facility_category
0,1,sector 25,4.35,5,4,3.0,old,1350.0,1,0,0
1,0,sector 50,3.10,3,4,4.0,old,2450.0,1,2,2
2,0,sector 63a,1.65,2,2,1.0,under_construction,978.0,0,0,0
3,0,sector 95,2.00,3,4,5.0,new,1964.0,1,0,0
4,0,sector 111,3.90,4,5,6.0,new,2996.0,1,1,2


In [11]:
new_df = pd.get_dummies(df, columns=['sector', 'agePossession'], drop_first=True)
new_df.head()


,property_type,price,bedRoom,bathroom,floorNum,builtup_area,servant room,furnishing_type,facility_category,sector_gwal pahari,sector_manesar,sector_new,sector_new sector 2,sector_sector 1,sector_sector 102,sector_sector 103,sector_sector 104,sector_sector 105,sector_sector 106,sector_sector 107,sector_sector 108,sector_sector 109,sector_sector 10a,sector_sector 11,sector_sector 110,sector_sector 111,sector_sector 112,sector_sector 113,sector_sector 12,sector_sector 13,sector_sector 14,sector_sector 15,sector_sector 17,sector_sector 17a,sector_sector 17b,sector_sector 2,sector_sector 21,sector_sector 22,sector_sector 23,sector_sector 24,sector_sector 25,sector_sector 26,sector_sector 27,sector_sector 28,sector_sector 3,sector_sector 3 phase 2,sector_sector 3 phase 3 extension,sector_sector 30,sector_sector 31,sector_sector 33,sector_sector 36,sector_sector 36a,sector_sector 37,sector_sector 37c,sector_sector 37d,sector_sector 38,sector_sector 39,sector_sector 4,sector_sector 40,sector_sector 41,sector_sector 43,sector_sector 45,sector_sector 46,sector_sector 47,sector_sector 48,sector_sector 49,sector_sector 5,sector_sector 50,sector_sector 51,sector_sector 52,sector_sector 53,sector_sector 54,sector_sector 55,sector_sector 56,sector_sector 57,sector_sector 58,sector_sector 59,sector_sector 6,sector_sector 60,sector_sector 61,sector_sector 62,sector_sector 63,sector_sector 63a,sector_sector 65,sector_sector 66,sector_sector 67,sector_sector 67a,sector_sector 68,sector_sector 69,sector_sector 7,sector_sector 70,sector_sector 70a,sector_sector 71,sector_sector 72,sector_sector 73,sector_sector 74,sector_sector 76,sector_sector 77,sector_sector 78,sector_sector 79,sector_sector 8,sector_sector 80,sector_sector 81,sector_sector 82,sector_sector 82a,sector_sector 83,sector_sector 84,sector_sector 85,sector_sector 86,sector_sector 88a,sector_sector 88b,sector_sector 89,sector_sector 9,sector_sector 90,sector_sector 91,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sector 99a,sector_sector 9a,sector_sohna road,sector_sohna road road,agePossession_old,agePossession_under_construction
0,1,4.35,5,4,3.0,1350.0,1,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,0,3.10,3,4,4.0,2450.0,1,2,2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,0,1.65,2,2,1.0,978.0,0,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False

In [12]:
X = new_df.drop(columns=['price'])
y = new_df['price']
y_log = np.log1p(y) # it was right skewed, so we apply log transformation

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

X_scaled.head()


,property_type,bedRoom,bathroom,floorNum,builtup_area,servant room,furnishing_type,facility_category,sector_gwal pahari,sector_manesar,sector_new,sector_new sector 2,sector_sector 1,sector_sector 102,sector_sector 103,sector_sector 104,sector_sector 105,sector_sector 106,sector_sector 107,sector_sector 108,sector_sector 109,sector_sector 10a,sector_sector 11,sector_sector 110,sector_sector 111,sector_sector 112,sector_sector 113,sector_sector 12,sector_sector 13,sector_sector 14,sector_sector 15,sector_sector 17,sector_sector 17a,sector_sector 17b,sector_sector 2,sector_sector 21,sector_sector 22,sector_sector 23,sector_sector 24,sector_sector 25,sector_sector 26,sector_sector 27,sector_sector 28,sector_sector 3,sector_sector 3 phase 2,sector_sector 3 phase 3 extension,sector_sector 30,sector_sector 31,sector_sector 33,sector_sector 36,sector_sector 36a,sector_sector 37,sector_sector 37c,sector_sector 37d,sector_sector 38,sector_sector 39,sector_sector 4,sector_sector 40,sector_sector 41,sector_sector 43,sector_sector 45,sector_sector 46,sector_sector 47,sector_sector 48,sector_sector 49,sector_sector 5,sector_sector 50,sector_sector 51,sector_sector 52,sector_sector 53,sector_sector 54,sector_sector 55,sector_sector 56,sector_sector 57,sector_sector 58,sector_sector 59,sector_sector 6,sector_sector 60,sector_sector 61,sector_sector 62,sector_sector 63,sector_sector 63a,sector_sector 65,sector_sector 66,sector_sector 67,sector_sector 67a,sector_sector 68,sector_sector 69,sector_sector 7,sector_sector 70,sector_sector 70a,sector_sector 71,sector_sector 72,sector_sector 73,sector_sector 74,sector_sector 76,sector_sector 77,sector_sector 78,sector_sector 79,sector_sector 8,sector_sector 80,sector_sector 81,sector_sector 82,sector_sector 82a,sector_sector 83,sector_sector 84,sector_sector 85,sector_sector 86,sector_sector 88a,sector_sector 88b,sector_sector 89,sector_sector 9,sector_sector 90,sector_sector 91,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sector 99a,sector_sector 9a,sector_sohna road,sector_sohna road road,agePossession_old,agePossession_under_construction
0,1.942572,1.561377,0.521101,-0.641426,-0.428495,1.339993,-0.684726,-0.985377,-0.069446,-0.093965,-0.033624,-0.037598,-0.041193,-0.176493,-0.109545,-0.138855,-0.075356,-0.104138,-0.130152,-0.129025,-0.157747,-0.044499,-0.067363,-0.080845,-0.079057,-0.080845,-0.110857,-0.095482,-0.047579,-0.071469,-0.041193,-0.041193,-0.016805,-0.029115,-0.132377,-0.041193,-0.075356,-0.060694,-0.075356,10.009995,-0.105515,-0.023769,-0.102744,-0.082596,-0.041193,-0.050472,-0.037598,-0.05321,-0.138855,-0.055815,-0.055815,-0.016805,-0.120868,-0.134568,-0.05321,-0.055815,-0.113438,-0.037598,-0.055815,-0.126745,-0.047579,-0.055815,-0.067363,-0.12559,-0.112154,-0.041193,-0.131269,-0.044499,-0.062994,-0.062994,-0.085993,-0.05321,-0.12325,-0.077228,-0.044499,-0.047579,-0.055815,-0.065214,-0.108217,-0.067363,-0.062994,-0.065214,-0.156805,-0.108217,-0.115965,-0.077228,-0.092424,-0.164208,-0.0999,-0.11844,-0.124425,-0.087643,-0.090857,-0.029115,-0.071469,-0.065214,-0.085993,-0.067363,-0.148079,-0.044499,-0.037598,-0.158685,-0.108217,-0.073438,-0.139907,-0.11844,-0.176493,-0.133477,-0.077228,-0.029115,-0.12789,-0.058305,-0.159617,-0.069446,-0.16957,-0.047579,-0.125590,-0.058305,-0.092424,-0.05321,-0.211749,-0.055815,1.654173,-0.290701
1,-0.514782,-0.067366,0.521101,-0.476411,0.527050,1.339993,1.556770,1.863770,-0.069446,-0.093965,-0.033624,-0.037598,-0.041193,-0.176493,-0.109545,-0.138855,-0.075356,-0.104138,-0.130152,-0.129025,-0.157747,-0.044499,-0.067363,-0.080845,-0.079057,-0.080845,-0.110857,-0.095482,-0.047579,-0.071469,-0.041193,-0.041193,-0.016805,-0.029115,-0.132377,-0.041193,-0.075356,-0.060694,-0.075356,-0.099900,-0.105515,-0.023769,-0.102744,-0.082596,-0.041193,-0.050472,-0.037598,-0.05321,-0.138855,-0.055815,-0.055815,-0.016805,-0.120868,-0.134568,-0.05321,-0.055815,-0.113438,-0.037598,-0.055815,-0.126745,-0.047579,-0.055815,-0.067363,-0.12559,-0.1121

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')


In [14]:
scores.mean(), scores.std()


(np.float64(0.8584695023553888), np.float64(0.018712122429398556))

In [15]:
lr = LinearRegression()

lr.fit(X_scaled, y_log)


LinearRegression()

In [16]:
lr.coef_


array([ 1.25894688e-01,  4.62583527e-02,  6.61747497e-02,  2.23853698e-02,
        2.15985093e-01,  4.67378954e-02,  5.77234774e-03,  4.31401962e-03,
        9.53057658e-03, -2.29663830e-02, -3.12403245e-03, -3.19244707e-03,
       -5.52258585e-03,  2.45127623e-02, -4.68598668e-03, -1.33467817e-03,
       -1.90780615e-02, -1.15918475e-03, -1.38589015e-02,  1.43462950e-02,
        2.19871410e-02,  4.78497389e-03, -1.49722033e-02,  9.29303146e-03,
        1.44987157e-02,  2.76848531e-02,  2.99538751e-02, -1.96243758e-02,
       -1.28109566e-02,  2.80177345e-02,  3.43419620e-03,  6.34442474e-03,
        4.20843929e-03,  1.25806652e-02,  6.11556353e-03, -9.04637662e-03,
        3.50075988e-02,  2.00276995e-03,  1.74068683e-02,  6.12894085e-02,
        7.32626200e-02,  7.06002312e-03,  4.23038914e-02, -1.24146074e-02,
       -1.06286981e-02, -1.39730167e-02,  6.63533360e-03,  2.29490529e-02,
        2.43568645e-02, -6.49633707e-03,  8.96845243e-03,  1.69316838e-03,
       -1.22179229e-02, -

In [17]:
X_scaled.shape


(3542, 124)

In [19]:
coef_df = pd.DataFrame(lr.coef_.reshape(1, 124), columns=X_scaled.columns).stack().reset_index().drop(columns='level_0').rename(columns={'level_1': 'feature', 0: 'coefficient'})
coef_df.head(5)


,feature,coefficient
0,property_type,0.125895
1,bedRoom,0.046258
2,bathroom,0.066175
3,floorNum,0.022385
4,builtup_area,0.215985


#### Regression Analysis

In [20]:
import statsmodels.api as sm

X_with_const = sm.add_constant(X_scaled)

model = sm.OLS(y_log, X_with_const).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.870
Model:                            OLS   Adj. R-squared:                  0.865
Method:                 Least Squares   F-statistic:                     184.5
Date:                Wed, 11 Jun 2025   Prob (F-statistic):               0.00
Time:                        18:13:44   Log-Likelihood:                 683.01
No. Observations:                3542   AIC:                            -1116.
Df Residuals:                    3417   BIC:                            -344.5
Df Model:                         124                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

## Standardized Regression Coefficients
**Formula:**  
```Stand. Coef (X) = UnStand. Coef (X) * (SD(X) / SD(Y))```  
we have the value of st. coef(x) and we want to find unstd. coef(x) value.  

therefore, formula becomes  
```UnStd. Coef(x) = Std. Coef.(X) * (SD(Y) / SD(X))```

In [ ]:
# get the st. deviation values of X and y

X_std = X_scaled['bedRoom'].std()
y_std = y_log.std()


unstd_coef_bedroom = 0.046258 * (y_std / X_std) # unstandardized coefficient for bedRoom

# need to consider exponent of the coefficient, because we applied log transformation on y
unstd_coef_bedroom_exp = np.expm1(unstd_coef_bedroom) 
unstd_coef_bedroom_exp


np.float64(0.025935795864130338)

In [24]:
# doing the same for builtup_area
X_std_builtup = X_scaled['builtup_area'].std()

unstd_coef_builtup = 0.215985 * (y_std / X_std_builtup) # unstandardized coefficient for builtup_area

unstd_coef_builtup_exp = np.expm1(unstd_coef_builtup)
unstd_coef_builtup_exp


np.float64(0.12699417819109354)